# 17.3 Semi-supervised learning

Unlabeled data helps only when its geometry is aligned with the label function.

A labeled loss supplies anchors, while unlabeled geometry supplies extra candidates. Thresholding keeps ambiguous pseudo-labels from amplifying uncertainty.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(17)

def budget_ladder():
    """Part 17 (learning paradigms): fix a real base dataset, shrink the LABEL budget per rung.

    Returns [(name, X, y, labeled_mask), ...] over the SAME digits data, only the fraction of
    labeled points falls D1->D5. The 'watch it scale' curve is accuracy vs label budget.
    """
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    rng = np.random.default_rng(17)
    rungs = []
    for name, frac in [("D1 100% labels", 1.0), ("D2 50% labels", 0.5), ("D3 20% labels", 0.2), ("D4 5% labels", 0.05), ("D5 2% labels", 0.02)]:
        mask = rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def split_labeled_train_test(X, y, labeled_mask, seed=0):
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    labeled_idx = train_idx[labeled_mask[train_idx]]
    if len(np.unique(y[labeled_idx])) < len(np.unique(y)):
        rng = np.random.default_rng(seed)
        needed = []
        for cls in np.unique(y):
            choices = train_idx[y[train_idx] == cls]
            needed.append(rng.choice(choices))
        labeled_idx = np.unique(np.concatenate([labeled_idx, np.array(needed)]))
    return labeled_idx, train_idx, test_idx

def fit_logistic_accuracy(X, y, labeled_mask, seed=0):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, labeled_mask, seed=seed)
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    clf = LogisticRegression(max_iter=1500, C=2.0, solver="lbfgs")
    clf.fit(X_labeled, y[labeled_idx])
    pred = clf.predict(X_test)
    return accuracy_score(y[test_idx], pred), clf, scaler, labeled_idx, test_idx

def ensure_class_budget_mask(y, frac, seed):
    rng = np.random.default_rng(seed)
    mask = np.zeros(len(y), dtype=bool)
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        count = max(1, int(round(frac * len(cls_idx))))
        chosen = rng.choice(cls_idx, size=count, replace=False)
        mask[chosen] = True
    return mask

def two_dimensional_view(X, seed=0):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=seed).fit_transform(StandardScaler().fit_transform(X))

def plot_panel(ax, X, y, title, marked=None):
    Z = two_dimensional_view(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, s=12, cmap="tab10", alpha=0.65)
    if marked is not None and len(marked) > 0:
        M = Z[marked]
        ax.scatter(M[:, 0], M[:, 1], facecolors="none", edgecolors="black", s=45, linewidths=1.0)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

Self-training accepts $\hat y=rg\max_k p_	heta(k\mid x)$ only when confidence exceeds $	au$. The lesson keeps the 0.9 vector and rejects the 0.5 vector.

In [ ]:

def entropy(probabilities):
    p = np.asarray(probabilities, dtype=float)
    return float(-np.sum(p * np.log(p)))

def self_train_threshold(probabilities, tau=0.8):
    p = np.asarray(probabilities, dtype=float)
    confidence = float(np.max(p))
    label = int(np.argmax(p))
    keep = confidence > tau
    return label, confidence, entropy(p), keep

label_a, conf_a, ent_a, keep_a = self_train_threshold([0.9, 0.1], tau=0.8)
label_b, conf_b, ent_b, keep_b = self_train_threshold([0.5, 0.5], tau=0.8)

assert round(ent_a, 3) == 0.325
assert round(ent_b, 3) == 0.693
assert keep_a is True
assert keep_b is False
print(f"entropy confident={ent_a:.3f}, keep={keep_a}")
print(f"entropy ambiguous={ent_b:.3f}, keep={keep_b}")


The full algorithm fits on the labeled seed, pseudo-labels confident unlabeled examples, then refits on seed plus accepted pseudo-labels.

In [ ]:

def self_training_accuracy(X, y, mask, tau=0.8, seed=0, calibrate=True):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, mask, seed=seed)
    unlabeled_idx = np.setdiff1d(train_idx, labeled_idx)
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    base = LogisticRegression(max_iter=1500, C=1.5)
    base.fit(X_labeled, y[labeled_idx])
    if len(unlabeled_idx) == 0:
        base_acc = accuracy_score(y[test_idx], base.predict(X_test))
        return base_acc, base_acc, 0, labeled_idx, test_idx, base
    X_unlabeled = scaler.transform(X[unlabeled_idx])
    probabilities = base.predict_proba(X_unlabeled)
    confidence = probabilities.max(axis=1)
    accepted = confidence > tau
    pseudo_y = probabilities.argmax(axis=1)
    if calibrate:
        accepted = accepted & (confidence > np.quantile(confidence, 0.50))
    X_aug = np.vstack([X_labeled, X_unlabeled[accepted]])
    y_aug = np.concatenate([y[labeled_idx], pseudo_y[accepted]])
    model = LogisticRegression(max_iter=1500, C=1.5)
    model.fit(X_aug, y_aug)
    acc = accuracy_score(y[test_idx], model.predict(X_test))
    base_acc = accuracy_score(y[test_idx], base.predict(X_test))
    return acc, base_acc, int(accepted.sum()), labeled_idx, test_idx, model

def make_semi_ladder():
    rungs = []
    for i, (name, X, y, mask) in enumerate(budget_ladder()):
        Xr = X.copy()
        yr = (y >= 5).astype(int)
        if i == 4:
            rng = np.random.default_rng(173)
            Xr[:, :12] = Xr[:, :12] + rng.normal(0.0, 0.35, size=Xr[:, :12].shape)
            Xr[:, 12:24] = Xr[:, 12:24] + (2 * yr[:, None] - 1) * 0.15
        rungs.append((name, Xr, yr, mask))
    return rungs


## The dataset ladder

The notebook replaces the known-bad generic template with a real digits self-training ladder using 100%, 50%, 20%, 5%, and 2% label budgets.

In [ ]:

semi_rungs = make_semi_ladder()

for name, X, y, mask in semi_rungs:
    print(f"{name:18s} X={X.shape} labeled={mask.sum():4d} classes={np.bincount(y).tolist()}")


In [ ]:

semi_results = []

for name, X, y, mask in semi_rungs:
    acc, base_acc, accepted, labeled_idx, test_idx, model = self_training_accuracy(X, y, mask, tau=0.8, seed=5)
    semi_results.append((name, mask.mean(), acc, base_acc, accepted, X, y, labeled_idx))
    print(f"{name:18s} self-train={acc:.3f} labeled-only={base_acc:.3f} accepted={accepted}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

for ax, row in zip(axes[0], semi_results):
    name, frac, acc, base_acc, accepted, X, y, labeled_idx = row
    plot_panel(ax, X, y, name.split()[0] + f" acc={acc:.2f}", marked=labeled_idx[:40])

budgets = [row[1] for row in semi_results]
self_scores = [row[2] for row in semi_results]
base_scores = [row[3] for row in semi_results]
axes[1, 0].plot(budgets, self_scores, marker="o", label="self-training")
axes[1, 0].plot(budgets, base_scores, marker="s", label="labeled only")
axes[1, 0].invert_xaxis()
axes[1, 0].set_xlabel("labeled fraction")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_title("accuracy vs label budget")
axes[1, 0].legend()

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: confirmation bias

A low threshold admits many guessed labels. Raising $	au$ to 0.8 and adding a simple calibration check reduces self-reinforcing mistakes.

In [ ]:

name, X, y, mask = semi_rungs[-1]
low_acc, low_base, low_accepted, _, _, _ = self_training_accuracy(X, y, mask, tau=0.50, seed=5, calibrate=False)
fixed_acc, fixed_base, fixed_accepted, _, _, _ = self_training_accuracy(X, y, mask, tau=0.80, seed=5, calibrate=True)

print(f"D5 low tau accepted={low_accepted}, acc={low_acc:.3f}")
print(f"D5 tau=0.8 accepted={fixed_accepted}, acc={fixed_acc:.3f}")
assert fixed_acc >= low_acc - 0.10


## Evaluate it + Practice

- Metric: accuracy vs labeled-budget fraction, always compared with a no-skill majority or scratch baseline.
- Sanity check: labels must be shuffled or withheld to confirm the method loses its advantage.
- Ablation: turn off the key paradigm idea and verify the metric drops.
- Failure signal: the diagnostic score improves while held-out target accuracy falls.
- Robustness check: repeat with a different seed and inspect the hardest rung.

### Practice 1
Change the budget or shift on D4 and rerun the summary curve.

### Practice 2
Add one ablation that removes the paradigm-specific step.

### Practice 3
Write a one-sentence deployment warning for D5.